In [35]:
!pip install pipenv
!pip install uv
!uv sync --locked
import pickle
import pandas as pd
import numpy as np

Resolved 19 packages in 2ms
Checked 18 packages in 0.46ms


In [3]:
!git clone --filter=blob:none --no-checkout --depth 1 https://github.com/DataTalksClub/machine-learning-zoomcamp.git
%cd machine-learning-zoomcamp
!git sparse-checkout set cohorts/2026/homework/05-deployment
!git checkout


Cloning into 'machine-learning-zoomcamp'...
remote: Enumerating objects: 138, done.
remote: Counting objects: 100% (138/138), done.
remote: Compressing objects: 100% (116/116), done.
remote: Total 138 (delta 0), reused 106 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (138/138), 63.21 KiB | 1.92 MiB/s, done.
/content/machine-learning-zoomcamp
remote: Enumerating objects: 25, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 25 (delta 1), reused 18 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (25/25), 37.99 KiB | 762.00 KiB/s, done.
Resolving deltas: 100% (1/1), done.
Your branch is up to date with 'origin/main'.


In [4]:
%cd cohorts/2026/homework/05-deployment

/content/machine-learning-zoomcamp/cohorts/2026/homework/05-deployment


In [5]:
!uv run python smoke_test.py

Using CPython 3.11.15
Creating virtual environment at: .venv
Installed 18 packages in 153ms
{'conversion_probability': 0.769799, 'conversion': True}


In [7]:
!sha256sum pipeline.bin

1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1  pipeline.bin


# Question 1 — environment check
Run uv --version and record the version in your notes. This is a setup check,not a graded answer: your local uv version is not a property of the model.

In [9]:
!uv --version

uv 0.13.5 (x86_64-unknown-linux-gnu)


# Question 2 — locked dependency
Open pyproject.toml and uv.lock. Which Scikit-Learn version is part of the 2026 reference environment?

- 1.6.1
- 1.7.2
- 1.8.0
- 2.0.0

Do not regenerate the lockfile for the graded run; use uv sync --locked.

#Answer
- 1.7.2

In [10]:
!cat pyproject.toml

[project]
name = "mlzoomcamp-2026-deployment"
version = "0.1.0"
description = "Reference deployment project for ML Zoomcamp 2026"
requires-python = ">=3.11,<3.12"
dependencies = [
    "fastapi==0.119.1",
    "numpy==2.3.3",
    "scikit-learn==1.7.2",
    "uvicorn==0.38.0",
]

[tool.uv]
exclude-newer = "2026-09-14T23:59:59Z"


# Question 3 — load the model
Write a Python script that loads pipeline.bin with pickle and calls predict_proba for this lead:

Report the probability of conversion rounded to three decimal places. The grader accepts an absolute error of at most 0.005; do not choose the nearest answer from a list.

In [33]:
with open ('pipeline.bin', 'rb') as f_in:
    dv, model = pickle.load(f_in)


lead = {
  "lead_source": "paid_ads",
  "industry": "technology",
  "employment_status": "employed",
  "location": "north_america",
  "number_of_courses_viewed": 2,
  "annual_income": 79276.0,
  "interaction_count": 4,
  "lead_score": 0.41
}

#df = pd.DataFrame([lead])
X = dv.transform(lead)
y_pred = model.predict_proba(X)[:,1]
print('prediction: %.3f'% y_pred)

prediction: 0.533


/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator DictVectorizer from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.7.2 when using version 1.6.1. This might lead to breaking code or in

# Question 4 — serve the model
Start the provided API:

Report conversion_probability rounded to three decimal places. The same 0.005 absolute tolerance applies. The response must contain a probability in [0, 1] and a boolean conversion field.

#Answer
"conversion_probability":0.769799,"conversion":

In [38]:
  !uv run uvicorn predict:app --host 0.0.0.0 --port 9696

INFO:     Started server process [34940]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:9696 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [34940]
